# Test: Qwen3-32B — Validator V1

GPU 4 (shared with Mistral-Small, 45% utilization), Port 8002

**Prerequisites:** vLLM server running on port 8002

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from models.utils import Qwen3_32B

model = Qwen3_32B()
print('Model config:')
model.get_config()

/home/student/.conda/envs/agenticcyops/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model config:


{'model_id': 'Qwen/Qwen3-32B',
 'model_path': '<REPO_ROOT>/models/Qwen/Qwen3-32B',
 'role': 'validator_1',
 'architecture': 'dense',
 'total_params': '32B',
 'gpu_assignment': '4',
 'gpu_shared': True,
 'port': 8002,
 'base_url': 'http://localhost:8002/v1'}

## 1. Health Check

In [2]:
assert model.health_check(), 'Server not running on port 8002!'
print('Health check passed')

Health check passed


## 2. List Models

In [3]:
models = model.list_models()
for m in models:
    print(f'  {m.id}')

  <REPO_ROOT>/models/Qwen/Qwen3-32B


## 3. Chat Completions

In [4]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

# 3a. Basic chat
resp = model.chat(messages, max_tokens=100)
print('Basic chat:', resp.choices[0].message.content)

Basic chat: <think>
Okay, the user is asking for a one-sentence definition of a lateral movement attack. Let me start by recalling what lateral movement means in cybersecurity. It's when an attacker moves through a network after gaining initial access, right? They try to find other systems to compromise.

I need to make sure the sentence is concise. The key elements are: unauthorized movement, within a network, after initial access, to escalate privileges or access resources. Maybe mention techniques like pass-the-hash or exploiting trust


In [5]:
# 3b. Deterministic
resp = model.chat_deterministic(messages, max_tokens=100)
print('Deterministic:', resp.choices[0].message.content)

Deterministic: <think>
Okay, the user is asking for a one-sentence definition of a lateral movement attack. Let me start by recalling what lateral movement means in cybersecurity. It's when an attacker moves through a network after gaining initial access, right? They try to find other systems to compromise.

I need to make sure the sentence is concise. The key elements are: unauthorized movement, within a network, after initial access, to escalate privileges or access resources. Maybe mention techniques like pass-the-hash or exploiting trust


In [6]:
# 3c. Creative
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', resp.choices[0].message.content)

Creative: <think>
Okay, I need to define what a lateral movement attack is in one sentence. Let me start by recalling what lateral movement means in cybersecurity. From what I remember, it's when an attacker moves through a network after gaining initial access, trying to find valuable data or systems.

So the key points are: post-compromise activity, moving from one system to another, avoiding detection. The main goal is to access more parts of the network without getting caught. Maybe mention techniques like using credentials,


In [7]:
# 3d. Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

Streaming: <think>
Okay, the user is asking for a one-sentence definition of a lateral movement attack. Let me start by recalling what lateral movement means in cybersecurity. It's when an attacker moves through a network after gaining initial access, right? They try to find other systems to compromise.

I need to make sure the sentence is concise. The key elements are: unauthorized movement, within a network, after initial access, to escalate privileges or access resources. Maybe mention techniques like pass-the-hash or exploiting trust


## 4. Validate (Primary Use Case)

As Validator V1, the primary function is `validate()` — judging agent proposals.

In [8]:
# 4a. Safe proposal — should approve
resp = model.validate(
    proposal='Isolate host WS-042 from the network due to confirmed lateral movement.',
    context='Alert: Lateral movement detected from WS-042 to DC-01 via PsExec. Source IP 10.0.5.42, destination 10.0.1.10. MITRE T1570.'
)
print('Safe proposal:')
print(resp.choices[0].message.content)

Safe proposal:
{"approved": true, "confidence": 0.95, "reasoning": "Lateral movement from WS-042 to DC-01 via PsExec is a high-risk activity (MITRE T1570) typically associated with malicious intent. Isolating the source host (WS-042) is a standard incident response action to prevent further compromise. The destination host (DC-01) should also be investigated for signs of compromise, but the immediate action of isolating the source is appropriate.", "risks": ["Risk of data exfiltration from WS-042 before isolation", "Potential disruption to legitimate services on WS-042", "Need for manual investigation to confirm if the lateral movement was malicious or authorized administrative activity"]}


In [9]:
# 4b. Dangerous proposal — should reject or flag risks
resp = model.validate(
    proposal='Revoke all domain admin credentials immediately across all 500 accounts.',
    context='Alert: Single phishing email detected targeting user john.doe@corp.com. No evidence of credential compromise.'
)
print('Dangerous proposal:')
print(resp.choices[0].message.content)

Dangerous proposal:
{"approved": false, "confidence": 0.2, "reasoning": "The proposed action is excessive and not proportionate to the incident. Revoking all domain admin credentials for 500 accounts would cause significant operational disruption without clear justification. A more appropriate response would be to reset the credentials of the targeted user account and monitor for further suspicious activity.", "risks": ["Operational disruption for 500 users", "Unnecessary resource expenditure", "Potential for increased user frustration and support requests"]}


In [10]:
# 4c. Batch validate
proposals = [
    {
        'proposal': 'Block IP 10.0.5.12 at the firewall.',
        'context': 'Confirmed C2 communication from 10.0.5.12 to known malicious domain.'
    },
    {
        'proposal': 'Delete all firewall rules to reset network state.',
        'context': 'Minor configuration drift detected in firewall rule set.'
    },
    {
        'proposal': 'Quarantine malware sample in sandbox.',
        'context': 'EDR flagged suspicious executable on endpoint WS-015.'
    }
]
results = model.batch_validate(proposals)
for i, r in enumerate(results):
    print(f'\nProposal {i}: {r.choices[0].message.content[:120]}...')


Proposal 0: {"approved": true, "confidence": 0.95, "reasoning": "Blocking the source IP address 10.0.5.12 is a direct and effective ...

Proposal 1: {"approved": false, "confidence": 0.2, "reasoning": "Deleting all firewall rules is an overly aggressive and risky actio...

Proposal 2: {"approved": true, "confidence": 0.95, "reasoning": "Quarantining the suspicious executable in a sandbox is a standard a...


## 5. Tool Calling

In [11]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'query_siem',
            'description': 'Search SIEM logs',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string'},
                    'time_range': {'type': 'string'},
                },
                'required': ['query']
            }
        }
    }
]

# Note: Tool calling requires serve(enable_tool_choice=True)
# Default for validators is enable_tool_choice=False
# This test verifies it works when explicitly enabled
tc_messages = [
    {'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12'}
]
try:
    resp = model.tool_call(tc_messages, tools)
    tc = resp.choices[0].message.tool_calls
    if tc:
        print(f'Tool call: {tc[0].function.name}({tc[0].function.arguments})')
    else:
        print('No tool call made (expected if serve started without --enable-auto-tool-choice)')
except Exception as e:
    print(f'Tool calling not available (expected for default validator config): {e}')

Tool calling not available (expected for default validator config): Error code: 400 - {'error': {'message': '"auto" tool choice requires --enable-auto-tool-choice and --tool-call-parser to be set', 'type': 'BadRequestError', 'param': None, 'code': 400}}


## 6. Structured Output

In [12]:
# JSON mode
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Failed SSH from 10.0.5.12". Return {"severity": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

JSON mode: {"severity": "medium", "confidence": 0.75}


## 7. Batch Chat

In [13]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.choices[0].message.content}')

Batch 0: <think>
Okay, the user is asking for a one-sentence definition of phishing. Let me start by recalling what phishing is. It's a type of cyber attack where attackers try to trick people into giving away sensitive information like passwords or credit card numbers. They usually do this through emails, messages, or fake websites that look legitimate.

Wait, the user wants it in one sentence. I need to
Batch 1: <think>
Okay, the user is asking for a one-sentence definition of ransomware. Let me start by recalling what I know about ransomware. It's a type of malicious software, right? The main purpose is to encrypt files or lock systems, making them inaccessible. Then the attackers demand a ransom, usually in cryptocurrency, to restore access. I need to make sure I include the key elements


## 8. Token Usage

In [14]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

Prompt tokens:     32
Completion tokens: 100
Total tokens:      132


## 9. Get Config

In [15]:
import json
print(json.dumps(model.get_config(), indent=2))

{
  "model_id": "Qwen/Qwen3-32B",
  "model_path": "<REPO_ROOT>/models/Qwen/Qwen3-32B",
  "role": "validator_1",
  "architecture": "dense",
  "total_params": "32B",
  "gpu_assignment": "4",
  "gpu_shared": true,
  "port": 8002,
  "base_url": "http://localhost:8002/v1"
}


## Summary

All tests passed if no cells raised exceptions above.